In [1]:
import sys
import os
from pathlib import Path
import yaml

In [2]:
# # 1. Ensure Python understands the project root so it can import modules from src
# project_root = Path(os.getcwd())
# # If running inside the notebooks folder, move up one level
# if project_root.name == "notebooks":
#     project_root = project_root.parent
# if str(project_root) not in sys.path:
#     sys.path.append(str(project_root))


In [3]:
from src.core.parser import HiveScriptParser
from src.transformers.basic_pyspark_transformer import BasicPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *

In [4]:
script_name = "com_t_mhbos_m_client"
datalake_type_subfolder = 'dml'
datalake_layer_subfolder = script_name.split("_")[0]

# sql_file_path = PROJECT_ROOT / "samples" / "input" / "ddl" / "raw" / f"{script_name}.sql"
sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.sql"
output_file_path = PROJECT_ROOT / "samples" / "converted" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"
variable_path = PROJECT_ROOT / "configs" / "rules" / "variable.yaml"

In [5]:
def transform_and_export(script_path, transformer_class):
    print("=== STARTING PIPELINE TEST RUN ===")
    #
    # script_path = Path(script_path)
    # script_name = script_path.name
    # script_parent = script_path.parent.name
    # script_grandparent = script_path.parent.parent.name

    # Path to the SQL file to parse
    sql_file_path = script_path

    # 3. Parse the SQL file into Context (Single Source of Truth)
    print(f"[2] Reading and parsing file: {sql_file_path.name}...")
    context = HiveScriptParser.parse_file(str(sql_file_path))
    print(f"    - Extracted Source: {context.source_name}, Table: {context.table_name}")
    print(f"    - Detected {len(context.ast_nodes)} SQL statements (AST nodes)")

    # 4. Initialize the Transformer and convert the AST structure
    print("[3] Starting AST transformation (variable wrapping, dialect conversion)...")
    transformer = transformer_class()
    render_model = transformer.transform(context)
    print(f"    - Is the table partitioned? -> {render_model.is_partitioned}")

    # 5. Render the data into the Jinja Template
    print("[4] Rendering data into the Jinja Template...")
    python_script = render_template(
        template_name="pyspark/pyspark_basic.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(python_script)

    print("\n" + "=" * 50)
    print(f"RESULT: COMPLETE PYTHON FILE SAVED AT {output_file_path}")
    print("=" * 50 + "\n")


In [ ]:
for 

In [7]:
def main():


    # Table name
    transform_and_export(sql_file_path, BasicPySparkTransformer)


if __name__ == "__main__":
    main()


=== STARTING PIPELINE TEST RUN ===
[2] Reading and parsing file: com_t_mhbos_m_client.sql...


'analyze table ${com_schema}.t_mhbos_m_client partition (etl_dt = '${batch_date}') compute statistics' contains unsupported syntax. Falling back to parsing as a 'Command'.


    - Extracted Source: t, Table: mhbos_m_client
    - Detected 120 SQL statements (AST nodes)
[3] Starting AST transformation (variable wrapping, dialect conversion)...
    - Is the table partitioned? -> False
[4] Rendering data into the Jinja Template...

RESULT: COMPLETE PYTHON FILE SAVED AT C:\Users\ext_giadung\projects\hql_spark_bridge\samples\converted\dml\com\com_t_mhbos_m_client.py

